# $\color{hotpink}{\text{Spectral with}}$

La función de densidad espectral de potencia para la técnica auto-heterodina con tiempos de retardo subcoherentes está dada por:

$
S(\omega, \tau) = \frac{\frac{1}{2} P_0^2 \tau_c}{1 + (\omega \pm \Omega)^2 \tau_c^2} \left\{ 1 - e^{-|\tau|/\tau_c} \cdot \left[ \cos (\omega \pm \Omega) |\tau| + \frac{\sin (\omega \pm \Omega) |\tau|}{(\omega \pm \Omega) \tau_c} \right] \right\} + \frac{1}{2} P_0^2 \pi e^{-|\tau|/\tau_c} \delta (\omega \pm \Omega)
$

donde:

   $S(\omega, \tau)$ es la densidad espectral de potencia,
   $P_0$ es la potencia de salida del láser,
   $\tau_c$ es el tiempo de coherencia del láser,
   $\tau$ es el tiempo de retardo de una ruta con respecto a la otra,
   $\Omega$ es la frecuencia de desplazamiento del modulador acusto-óptico (AOM),
   $\omega$ es la frecuencia angular,
   $\delta(\cdot)$ es la función delta de Dirac.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from scipy.signal import welch
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo para las gráficas
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
def power_spectrum(f, tau, tau_c, P0=1, Omega=54e6):
    """
    Calcula el espectro de potencia según la ecuación del artículo
    
    Parámetros:
    f: frecuencia (Hz)
    tau: tiempo de retardo (s)
    tau_c: tiempo de coherencia (s)
    P0: potencia óptica (W)
    Omega: frecuencia de desplazamiento AOM (Hz)
    """
    omega = 2 * np.pi * f
    
    # Término principal (parte Lorentziana modificada)
    term1 = (0.5 * P0**2 * tau_c) / (1 + (omega - 2*np.pi*Omega)**2 * tau_c**2)
    
    # Término de oscilaciones
    cos_term = np.cos((omega - 2*np.pi*Omega) * np.abs(tau))
    sin_term = np.sin((omega - 2*np.pi*Omega) * np.abs(tau)) / ((omega - 2*np.pi*Omega) * tau_c)
    
    # Evitar división por cero
    if np.any(np.isnan(sin_term)):
        sin_term = np.where(np.isnan(sin_term), np.abs(tau)/tau_c, sin_term)
    
    bracket_term = 1 - np.exp(-np.abs(tau)/tau_c) * (cos_term + sin_term)
    
    # Término delta (aproximamos con una Lorentziana muy estrecha)
    delta_width = 10  # Hz (para aproximar la función delta)
    delta_term = (0.5 * P0**2 * np.pi * np.exp(-np.abs(tau)/tau_c) * 
                 (delta_width / (np.pi * (delta_width**2 + (omega - 2*np.pi*Omega)**2))))
    
    
    return term1 * bracket_term + delta_term

# Parámetros del láser (basados en el artículo)
tau_c = 5e-6  # tiempo de coherencia de 5 μs (corresponde a ~63 kHz de ancho de línea)
Omega = 54e6  # frecuencia del AOM (54 MHz)

# Diferientes tiempos de retardo (valores usados en el artículo)
tau_values = [1.25 * tau_c, 2.5 * tau_c, 5.0 * tau_c, 30.0 * tau_c]
tau_labels = [r'$|\tau|/\tau_c = 1.25$', 
              r'$|\tau|/\tau_c = 2.5$', 
              r'$|\tau|/\tau_c = 5.0$', 
              r'$|\tau|/\tau_c = 30.0$']

# Rango de frecuencias para visualización
freqs = np.linspace(53.5e6, 54.5e6, 1000)  # alrededor de 54 MHz

# Crear la figura con subplots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (tau, label) in enumerate(zip(tau_values, tau_labels)):
    # Calcular el espectro de potencia
    spectrum = power_spectrum(freqs, tau, tau_c)
    
    # Normalizar y convertir a dB
    spectrum_db = 10 * np.log10(spectrum / np.max(spectrum))
    
    # Graficar
    axes[i].plot(freqs/1e6, spectrum_db, 'b-', linewidth=1.5, color='hotpink')
    axes[i].set_xlabel('Frecuencia (MHz)')
    axes[i].set_ylabel('Amplitud (dB)')
    axes[i].set_title(f'Espectro de Potencia: {label}')
    axes[i].grid(True, alpha=0.3)
    axes[i].set_xlim(53.5, 54.5)
    axes[i].set_ylim(-30, 10)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
from scipy.special import erf
import matplotlib.pyplot as plt

def power_spectral_density(f, tau, tau_c, P0, Omega, delta_width=10):
    """
    Power spectral density function for self-heterodyne measurements
    
    Parameters:
    f: frequency array (Hz)
    tau: delay time (s)
    tau_c: coherence time (s)
    P0: laser output power (W)
    Omega: AOM offset frequency (Hz)
    delta_width: width of the delta function approximation (Hz)
    
    Returns:
    S: power spectral density
    """
    omega = 2 * np.pi * f
    Omega_rad = 2 * np.pi * Omega
    
    # Main Lorentzian term
    term1 = (0.5 * P0**2 * tau_c) / (1 + (omega - Omega_rad)**2 * tau_c**2)
    
    # Oscillation terms
    cos_term = np.cos((omega - Omega_rad) * np.abs(tau))
    
    # Handle division by zero carefully
    with np.errstate(divide='ignore', invalid='ignore'):
        sin_term = np.sin((omega - Omega_rad) * np.abs(tau)) / ((omega - Omega_rad) * tau_c)
        sin_term = np.nan_to_num(sin_term, nan=np.abs(tau)/tau_c)
    
    bracket_term = 1 - np.exp(-np.abs(tau)/tau_c) * (cos_term + sin_term)
    
    # Delta function term (approximated with a narrow Lorentzian)
    delta_term = (0.5 * P0**2 * np.pi * np.exp(-np.abs(tau)/tau_c) * 
                 (delta_width / (np.pi * (delta_width**2 + (omega - Omega_rad)**2))))
    
    return term1 * bracket_term + delta_term

def nonlinear_fit_psd(f_data, s_data, tau, initial_params, bounds=(-np.inf, np.inf), weights=None):
    """
    Nonlinear fitting function for power spectral density similar to Mathematica's NonlinearModelFit
    
    Parameters:
    f_data: frequency data (Hz)
    s_data: power spectral density data
    tau: delay time (s)
    initial_params: initial guess for parameters [tau_c, P0, Omega, delta_width]
    bounds: parameter bounds for optimization
    weights: weights for data points
    
    Returns:
    popt: optimized parameters
    pcov: parameter covariance matrix
    fit_function: fitted function
    """
    # Define the function to fit
    def fit_function(f, tau_c, P0, Omega, delta_width):
        return power_spectral_density(f, tau, tau_c, P0, Omega, delta_width)
    
    # Perform the nonlinear fit
    popt, pcov = curve_fit(
        fit_function, 
        f_data, 
        s_data, 
        p0=initial_params,
        bounds=bounds,
        sigma=weights,
        maxfev=10000  # Increase maximum function evaluations
    )
    
    # Create a function with the optimized parameters
    def optimized_function(f):
        return fit_function(f, *popt)
    
    return popt, pcov, optimized_function

# Example usage and visualization
if __name__ == "__main__":
    # Generate sample data
    f = np.linspace(53.5e6, 54.5e6, 1000)  # Frequency around 54 MHz
    tau = 3e-6  # 3 μs delay
    true_params = [5e-6, 1.0, 54e6, 10]  # [tau_c, P0, Omega, delta_width]
    
    # Calculate true PSD
    s_true = power_spectral_density(f, tau, *true_params)
    
    # Add noise to simulate experimental data
    np.random.seed(42)  # For reproducibility
    noise = 0.05 * np.max(s_true) * np.random.normal(size=len(f))
    s_data = s_true + noise
    
    # Set initial guess and bounds
    initial_guess = [4e-6, 0.8, 53.9e6, 5]  # Slightly off from true values
    param_bounds = ([1e-7, 0.1, 53.5e6, 1], [1e-4, 2.0, 54.5e6, 100])  # [lower bounds], [upper bounds]
    
    # Perform the nonlinear fit
    popt, pcov, fit_func = nonlinear_fit_psd(f, s_data, tau, initial_guess, bounds=param_bounds)
    
    # Calculate fitted values
    s_fit = fit_func(f)
    
    # Print results
    print("True parameters:", true_params)
    print("Fitted parameters:", popt)
    print("Parameter errors:", np.sqrt(np.diag(pcov)))
    
    # Calculate R-squared
    ss_res = np.sum((s_data - s_fit) ** 2)
    ss_tot = np.sum((s_data - np.mean(s_data)) ** 2)
    r_squared = 1 - (ss_res / ss_tot)
    print(f"R-squared: {r_squared:.6f}")
    
    # Plot results
    plt.figure(figsize=(10, 6))
    plt.plot(f/1e6, 10*np.log10(s_true/np.max(s_true)), 'b-', label='True PSD', linewidth=2)
    plt.plot(f/1e6, 10*np.log10(s_data/np.max(s_data)), 'ko', label='Noisy Data', markersize=3, alpha=0.5)
    plt.plot(f/1e6, 10*np.log10(s_fit/np.max(s_fit)), 'r--', label='Fitted PSD', linewidth=2)
    plt.xlabel('Frequency (MHz)')
    plt.ylabel('Normalized Power (dB)')
    plt.title('Nonlinear Fit of Power Spectral Density')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # Residual plot
    residuals = s_data - s_fit
    plt.figure(figsize=(10, 4))
    plt.plot(f/1e6, residuals, 'ko', markersize=3, alpha=0.5)
    plt.axhline(y=0, color='r', linestyle='--')
    plt.xlabel('Frequency (MHz)')
    plt.ylabel('Residuals')
    plt.title('Residuals of Nonlinear Fit')
    plt.grid(True, alpha=0.3)
    plt.show()